# Using earthaccess plugins: Harmony, PyDAP and SlideRule

`earthaccess` plugins reach services that are not part of the core library. Each
one is a small package that installs into the `earthaccess.plugins` namespace.

Once installed, `ea.plugins` lists them and dispatches calls to them:

| Plugin | Service | Returns | Use it for |
| --- | --- | --- | --- |
| `ea.plugins.harmony` | NASA Harmony | `list[Asset]` | subset / reproject / reformat in the cloud |
| `ea.plugins.pydap` | OPeNDAP (PyDAP) | lazy `xarray.Dataset` | read slices without downloading |
| `ea.plugins.sliderule` | SlideRule | `GeoDataFrame` | on-demand ICESat-2 / GEDI processing |

This notebook runs all three against real data and shows the results coming back.

## Setup

The plugins are not on PyPI yet, so install them from a checkout:

```bash
pip install -e ".[all]" \
  -e ~/github/betolink/earthaccess-plugin-pydap \
  -e ~/github/betolink/earthaccess-plugin-harmony \
  -e ~/github/betolink/earthaccess-plugin-sliderule \
  jupyterlab matplotlib
```

To read the NetCDF results with the `h5netcdf` engine, make sure it is recent
(`h5netcdf` < 1.7 is incompatible with `h5py` 3.16): `pip install -U h5netcdf`.

In [ ]:
import earthaccess as ea

ea.login()  # Earthdata Login; use strategy="interactive" if netrc is not set up

`list_plugins()` reads distribution metadata only (it never imports a
backend), and shows every known plugin, installed or not.

In [ ]:
ea.list_plugins(details=True)

## Harmony: subset a MUR SST granule over the Gulf of Mexico

Harmony is NASA's asynchronous, cloud-based transformation service. We search Sea
Surface Temperature granules, then ask Harmony to subset them to a Gulf of Mexico
box. `request()` returns immediately; the job runs remotely.

In [ ]:
gulf = (-98.0, 18.0, -80.0, 31.0)  # west, south, east, north

sst = ea.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    bounding_box=gulf,
    temporal=("2023-06-01", "2023-06-08"),
    count=2,
)
len(sst)

In [ ]:
# Does Harmony have a service for this collection?
ea.plugins.harmony.supports(sst)

`supports()` returns a `Capabilities` object: whether the collection is
supported plus the operations and output formats the service advertises.
`capabilities()` is an alias and also accepts a collection short name.

In [ ]:
ea.plugins.harmony.capabilities("ATL06")

In [ ]:
job = ea.plugins.harmony.request(sst, bbox=gulf)
job  # <HarmonyJob harmony:... PENDING>

The job is resumable and can be inspected or steered while it runs. Large
requests wait in Harmony's preview state (`JobStatus.PAUSED`); `skip_preview()`
accepts them, and `pause()` / `resume()` / `cancel()` control the order.

In [ ]:
job.status()
job.info()  # includes an order_url; never the request payload

In [ ]:
# Open the transformed results (blocks until the order finishes), or download them:
subsetted_files = ea.open(job)
# ea.download(job, "./data")
subsetted_files

In [ ]:
import xarray as xr

# combine="nested" is required when concatenating along a specific dimension.
ds = xr.open_mfdataset(subsetted_files, combine="nested", concat_dim="time")
ds["analysed_sst"].isel(time=0).plot(figsize=(10, 6), cmap="RdYlBu_r")

## PyDAP: read a MUR SST slice with OPeNDAP

OPeNDAP serves slices of a remote dataset on demand. We reuse the same MUR SST
search and open a granule lazily as an `xarray.Dataset`, then subset to the Gulf
of Mexico. Nothing is downloaded until we touch the data.

In [ ]:
mursst = ea.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    temporal=("2023-06-01", "2023-06-02"),
    count=1,
)
ds = ea.plugins.pydap.request(mursst, protocol="dap4").results()
ds

In [ ]:
# request(engine=...) keeps the same flow: engine="pydap" returns the native
# pydap dataset instead of xarray, so you can navigate the file and its
# operations (a metadata request, no data download).
ea.plugins.pydap.request(mursst, engine="pydap").results().tree()

In [ ]:
# Server-side slice: only the Gulf of Mexico window is fetched. A single
# granule is kept lazy, so the subset is pushed to the server.
gulf_sst = ds["analysed_sst"].isel(time=0).sel(lon=slice(-98, -80), lat=slice(18, 31))
gulf_sst.plot(figsize=(10, 6), cmap="RdYlBu_r")

For **several** granules `request()` concatenates them with `combine="nested"` +
`parallel=True` and dask. Chunk to your subset so only the needed slice is
fetched (otherwise dask pulls whole files):

```python
ds = ea.plugins.pydap.request(many_granules, chunks={"lat": 400, "lon": 400}).results()
```

## SlideRule: on-demand processing of ATL03 with a small bbox

SlideRule processes ICESat-2 and GEDI data in the cloud. We search a small ATL03
region, then run the `atl03x` processor over an area of interest. SlideRule
manages its own credentials — set your organization below.

In [ ]:
ea.plugins.sliderule.configure(
    url="slideruleearth.io",
    organization="sliderule",  # <- use your own SlideRule organization
)

In [ ]:
bbox = (-43.5, 66.5, -43.3, 66.7)  # a small ATL03 region

atl03 = ea.search_data(
    short_name="ATL03",
    bounding_box=bbox,
    temporal=("2023-06-01", "2023-06-30"),
    count=1,
)
len(atl03)

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [[-43.5, 66.5], [-43.3, 66.5], [-43.3, 66.7], [-43.5, 66.7], [-43.5, 66.5]]
    ],
}

df = ea.plugins.sliderule.run(atl03, api="atl03x", parms={}, aoi=aoi)
df.shape

In [ ]:
df.head()

In [ ]:
df.plot(column="height", markersize=0.5)  # geolocated points returned by SlideRule

## Which plugins accept a dataset?

`results.list_services()` asks every installed plugin whether it can handle the
collection, and calling a plugin that cannot fails early with a clear error.

In [ ]:
atl03.list_services()

In [ ]:
# The same information per plugin; capabilities() is an alias of supports() and
# also accepts a short name.
ea.plugins.sliderule.capabilities("ATL03")

In [ ]:
# SlideRule only supports ICESat-2 and GEDI collections; this raises
# UnsupportedCollectionError before any backend call.
from earthaccess.plugins import UnsupportedCollectionError

try:
    ea.plugins.sliderule.run(sst, api="atl03x")
except UnsupportedCollectionError as exc:
    print(exc)

## Summary

- `ea.plugins.<name>.request(results, **options)` returns a `Job` immediately.
- `ea.open(job)` / `ea.download(job, path)` consume asset jobs; xarray and
  GeoDataFrame jobs are returned as usable objects.
- Harmony jobs are remote and resumable (`get_job`, `pause`, `resume`,
  `skip_preview`, `cancel`).
- `ea.plugins.<name>.supports(results)` and `results.list_services()` tell you
  which plugins accept a dataset before you submit.